## 运行环境说明

### 虚拟环境设置

本项目使用Python虚拟环境管理依赖。在运行此演示之前，请确保已正确设置虚拟环境：

```bash
# 在项目根目录下
cd crypto_index_study

# 创建虚拟环境（如果还没有）
python -m venv .venv

# 激活虚拟环境
# Windows PowerShell:
.venv\Scripts\Activate.ps1
# Windows CMD:
.venv\Scripts\activate.bat
# Linux/Mac:
source .venv/bin/activate

# 安装依赖
pip install -r requirements.txt
```

### 检查环境状态

运行以下代码可以检查当前的Python环境：

In [ ]:
# 环境检查
import sys
from pathlib import Path

print("🔍 Python环境检查:")
print(f"Python版本: {sys.version}")
print(f"Python路径: {sys.executable}")
print(f"虚拟环境: {'✅ 是' if hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix) else '❌ 否'}")

# 检查是否在正确的项目目录
current_dir = Path.cwd()
project_markers = ['.git', 'src', 'data', 'scripts']
is_in_project = all((current_dir / marker).exists() for marker in project_markers[:2])
print(f"项目目录: {'✅ 正确' if is_in_project else '❌ 不在项目根目录'}")

print(f"当前目录: {current_dir}")
print("=" * 50)

🔍 Python环境检查:
Python版本: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python路径: d:\Codes\crypto_index_study\.venv\Scripts\python.exe
虚拟环境: ✅ 是
项目目录: ❌ 不在项目根目录
当前目录: d:\Codes\crypto_index_study\case_studies\01_Index_Calculation_Algorithm_Demo


# 加密货币指数计算演示

本演示将逐步展示如何构建和计算加密货币市值加权指数。

## 市值加权指数算法原理

### 什么是市值加权指数？

市值加权指数（Market Capitalization Weighted Index）是一种常见的指数编制方法，广泛应用于股票市场和加密货币市场。在这种方法中，每个成分资产在指数中的权重由其市值大小决定。

### 核心概念

**市值（Market Capitalization）**：指某个资产的总市场价值，计算公式为：
- 市值 = 当前价格 × 流通供应量

**权重（Weight）**：指某个资产在指数中所占的比例，计算公式为：
- 权重 = 该资产市值 / 所有成分资产市值总和

### 算法基本流程

1. **确定成分资产**：选择要纳入指数的资产（如前10大、前20大等）
2. **计算各资产市值**：获取每个资产的当前价格和流通供应量
3. **计算权重**：根据市值占比确定每个资产的权重
4. **设定基准值**：选择一个基准日期和基准指数值（通常为1000或100）
5. **计算指数值**：根据权重和价格变动计算当前指数值

### 指数计算公式

**基准日指数值**：
```
指数值 = 基准值（如1000）
```

**后续日期指数值**：
```
指数值 = 基准值 × (当前总市值 / 基准日总市值)
```

其中，总市值是所有成分资产市值的加权平均。

### 市值加权的优势

1. **客观性**：权重完全基于市场价值，无需主观判断
2. **代表性**：大市值资产对指数影响更大，更能反映市场整体状况
3. **可投资性**：投资者可以按权重比例构建投资组合
4. **广泛认可**：标普500、纳斯达克等主要指数都采用此方法

### 市值加权的局限性

1. **集中度风险**：大市值资产过度影响指数表现
2. **价格偏差**：可能放大泡沫或高估资产的影响
3. **波动性**：大市值资产的剧烈波动会显著影响指数

### 在加密货币中的应用

加密货币市值加权指数通常会：
- 排除稳定币（如USDT、USDC）
- 排除包装币和衍生品（如WBTC、stETH）
- 只包含原生代币
- 定期调整成分币种以反映市场变化

In [2]:
# 市值加权指数计算实例演示
# 基准日：2020年1月1日，基准值：1000
# 目标日：2021年12月31日

import pandas as pd
from pathlib import Path
import traceback
import sys

# 获取项目根目录
project_root = Path.cwd()
while not (project_root / ".git").exists() and project_root.parent != project_root:
    project_root = project_root.parent

print(f"🔍 项目根目录: {project_root}")
print(f"🔍 当前工作目录: {Path.cwd()}")

# 将项目根目录添加到Python路径
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"✅ 已将项目根目录添加到Python路径")

# 导入项目中已有的功能
try:
    from src.utils.data_readers import read_daily_snapshot, get_coin_info
    print("✅ 成功导入 data_readers 模块")
except Exception as e:
    print("❌ 导入 data_readers 模块失败:")
    print(f"   错误类型: {type(e).__name__}")
    print(f"   错误信息: {str(e)}")
    traceback.print_exc()
    sys.exit(1)

try:
    from src.classification.unified_classifier import UnifiedClassifier
    print("✅ 成功导入 UnifiedClassifier 模块")
except Exception as e:
    print("❌ 导入 UnifiedClassifier 模块失败:")
    print(f"   错误类型: {type(e).__name__}")
    print(f"   错误信息: {str(e)}")
    traceback.print_exc()
    sys.exit(1)

def filter_native_coins(df):
    """使用统一分类器过滤原生币种"""
    if df.empty:
        return df
    
    try:
        classifier = UnifiedClassifier()
        print(f"✅ 成功创建 UnifiedClassifier 实例")
        
        # 过滤出原生币种
        native_coins = []
        total_coins = len(df['coin_id'].unique())
        print(f"🔍 开始分类 {total_coins} 个币种...")
        
        for i, coin_id in enumerate(df['coin_id'].unique()):
            try:
                if classifier.is_native_coin(coin_id):
                    native_coins.append(coin_id)
                if (i + 1) % 50 == 0:
                    print(f"   已处理 {i + 1}/{total_coins} 个币种")
            except Exception as e:
                print(f"⚠️ 分类币种 {coin_id} 时出错:")
                print(f"   错误类型: {type(e).__name__}")
                print(f"   错误信息: {str(e)}")
                traceback.print_exc()
                continue
        
        print(f"✅ 分类完成，找到 {len(native_coins)} 个原生币种")
        
        # 返回原生币种且市值大于0的数据
        result = df[df['coin_id'].isin(native_coins) & (df['market_cap'] > 0)].copy()
        print(f"✅ 过滤完成，保留 {len(result)} 条记录")
        return result
        
    except Exception as e:
        print("❌ filter_native_coins 函数执行失败:")
        print(f"   错误类型: {type(e).__name__}")
        print(f"   错误信息: {str(e)}")
        traceback.print_exc()
        return df

# 步骤1：加载基准日期数据（2020年1月1日）
BASE_DATE = "2020-01-01"
BASE_VALUE = 1000

print("=" * 60)
print(f"📅 基准日期: {BASE_DATE}")
print(f"🎯 基准指数值: {BASE_VALUE}")
print("=" * 60)

try:
    data_path = str(project_root / "data/daily/daily_files")
    print(f"🔍 数据路径: {data_path}")
    
    base_df = read_daily_snapshot(BASE_DATE, data_path)
    print(f"✅ 成功调用 read_daily_snapshot")
    
    if base_df.empty:
        print(f"❌ 没有找到 {BASE_DATE} 的数据")
    else:
        print(f"✅ 成功加载 {BASE_DATE} 数据，共 {len(base_df)} 条记录")
        print(f"🔍 数据列: {list(base_df.columns)}")
        
        # 过滤原生币种
        base_filtered = filter_native_coins(base_df)
        
        if base_filtered.empty:
            print("❌ 过滤后没有原生币种数据")
        else:
            # 选择前10大币种作为指数成分
            top_10_base = base_filtered.nlargest(10, 'market_cap')
            
            print(f"✅ 成功加载 {BASE_DATE} 数据")
            print(f"📊 过滤后原生币种: {len(base_filtered)} 个")
            print(f"🏆 指数成分: 前10大原生币种")
            print()
            
            # 显示基准日成分
            print("📋 基准日指数成分:")
            print("-" * 60)
            base_total_market_cap = 0
            for rank, (_, row) in enumerate(top_10_base.iterrows(), 1):
                try:
                    metadata_path = str(project_root / "data/metadata/coin_metadata")
                    symbol, name = get_coin_info(row['coin_id'], metadata_path)
                    market_cap = row['market_cap']
                    base_total_market_cap += market_cap
                    weight = (market_cap / top_10_base['market_cap'].sum()) * 100
                    print(f"{rank:2d}. {symbol:<6} {name:<15} {market_cap:>15,.0f} ({weight:5.1f}%)")
                except Exception as e:
                    print(f"⚠️ 获取币种 {row['coin_id']} 信息时出错:")
                    print(f"   错误类型: {type(e).__name__}")
                    print(f"   错误信息: {str(e)}")
                    traceback.print_exc()
                    continue
            
            print("-" * 60)
            print(f"{'总市值':<25} {base_total_market_cap:>15,.0f}")
            print(f"{'基准指数值':<25} {BASE_VALUE:>15.0f}")
            print()

except Exception as e:
    print("❌ 加载基准日期数据时出错:")
    print(f"   错误类型: {type(e).__name__}")
    print(f"   错误信息: {str(e)}")
    traceback.print_exc()
    sys.exit(1)

# 步骤2：加载目标日期数据（2021年12月31日）
TARGET_DATE = "2021-12-31"

print("=" * 60)
print(f"📅 目标日期: {TARGET_DATE}")
print("=" * 60)

try:
    target_df = read_daily_snapshot(TARGET_DATE, data_path)
    print(f"✅ 成功调用 read_daily_snapshot")
    
    if target_df.empty:
        print(f"❌ 没有找到 {TARGET_DATE} 的数据")
    else:
        print(f"✅ 成功加载 {TARGET_DATE} 数据，共 {len(target_df)} 条记录")
        
        # 过滤原生币种
        target_filtered = filter_native_coins(target_df)
        
        print(f"✅ 成功加载 {TARGET_DATE} 数据")
        print(f"📊 过滤后原生币种: {len(target_filtered)} 个")
        print()
        
        # 计算目标日期时基准成分的市值
        target_constituents = target_filtered[target_filtered['coin_id'].isin(top_10_base['coin_id'])]
        
        print("📋 目标日期成分市值:")
        print("-" * 60)
        target_total_market_cap = 0
        for rank, (_, base_row) in enumerate(top_10_base.iterrows(), 1):
            try:
                coin_id = base_row['coin_id']
                symbol, name = get_coin_info(coin_id, metadata_path)
                
                # 查找目标日期的市值
                target_row = target_constituents[target_constituents['coin_id'] == coin_id]
                if not target_row.empty:
                    target_market_cap = target_row.iloc[0]['market_cap']
                    target_total_market_cap += target_market_cap
                    base_market_cap = base_row['market_cap']
                    change = ((target_market_cap - base_market_cap) / base_market_cap) * 100
                    print(f"{rank:2d}. {symbol:<6} {name:<15} {target_market_cap:>15,.0f} ({change:+6.1f}%)")
                else:
                    print(f"{rank:2d}. {symbol:<6} {name:<15} {'数据缺失':>15}")
            except Exception as e:
                print(f"⚠️ 处理币种 {coin_id} 时出错:")
                print(f"   错误类型: {type(e).__name__}")
                print(f"   错误信息: {str(e)}")
                traceback.print_exc()
                continue
        
        print("-" * 60)
        print(f"{'总市值':<25} {target_total_market_cap:>15,.0f}")
        
        # 步骤3：计算指数值
        if target_total_market_cap > 0:
            try:
                # 指数值 = 基准值 × (目标日总市值 / 基准日总市值)
                index_value = BASE_VALUE * (target_total_market_cap / base_total_market_cap)
                total_return = ((index_value - BASE_VALUE) / BASE_VALUE) * 100
                
                print(f"{'指数值':<25} {index_value:>15.2f}")
                print(f"{'总收益率':<25} {total_return:>14.1f}%")
                print()
                
                print("=" * 60)
                print("📊 计算过程:")
                print(f"指数值 = {BASE_VALUE} × ({target_total_market_cap:,.0f} / {base_total_market_cap:,.0f})")
                print(f"指数值 = {BASE_VALUE} × {target_total_market_cap/base_total_market_cap:.4f}")
                print(f"指数值 = {index_value:.2f}")
                print("=" * 60)
                
                # 年化收益率（约2年时间）
                years = 2
                annual_return = (((index_value / BASE_VALUE) ** (1/years)) - 1) * 100
                print(f"📈 年化收益率: {annual_return:.1f}%")
                
            except Exception as e:
                print("❌ 计算指数值时出错:")
                print(f"   错误类型: {type(e).__name__}")
                print(f"   错误信息: {str(e)}")
                traceback.print_exc()
        else:
            print("❌ 无法计算指数值：目标日期数据不足")

except Exception as e:
    print("❌ 加载目标日期数据时出错:")
    print(f"   错误类型: {type(e).__name__}")
    print(f"   错误信息: {str(e)}")
    traceback.print_exc()

print("=" * 60)
print("🎉 演示完成！")
print("=" * 60)

🔍 项目根目录: d:\Codes\crypto_index_study
🔍 当前工作目录: d:\Codes\crypto_index_study\case_studies\01_Index_Calculation_Algorithm_Demo
✅ 已将项目根目录添加到Python路径


2025-07-16 14:21:43,942 - BatchDownloader.2038359956944 - INFO - 批量下载器初始化完成
INFO:BatchDownloader.2038359956944:批量下载器初始化完成
2025-07-16 14:21:43,972 - BatchDownloader.2039192515920 - INFO - 批量下载器初始化完成
INFO:BatchDownloader.2039192515920:批量下载器初始化完成


✅ 成功导入 data_readers 模块
✅ 成功导入 UnifiedClassifier 模块
📅 基准日期: 2020-01-01
🎯 基准指数值: 1000
🔍 数据路径: d:\Codes\crypto_index_study\data\daily\daily_files
✅ 成功调用 read_daily_snapshot
✅ 成功加载 2020-01-01 数据，共 142 条记录
🔍 数据列: ['timestamp', 'price', 'volume', 'market_cap', 'date', 'coin_id', 'rank']
✅ 成功创建 UnifiedClassifier 实例
🔍 开始分类 142 个币种...
   已处理 50/142 个币种
   已处理 100/142 个币种
✅ 分类完成，找到 142 个原生币种
✅ 过滤完成，保留 142 条记录
✅ 成功加载 2020-01-01 数据
📊 过滤后原生币种: 142 个
🏆 指数成分: 前10大原生币种

📋 基准日指数成分:
------------------------------------------------------------
 1. BTC    Bitcoin         130,394,101,536 ( 76.3%)
 2. ETH    Ethereum         14,097,451,632 (  8.3%)
 3. XRP    XRP               8,359,480,714 (  4.9%)
 4. USDT   Tether            4,285,941,695 (  2.5%)
 5. BCH    Bitcoin Cash      3,721,681,728 (  2.2%)
 6. LTC    Litecoin          2,631,033,888 (  1.5%)
 7. EOS    EOS               2,475,835,361 (  1.4%)
 8. BNB    BNB               2,103,940,815 (  1.2%)
 9. BSV    Bitcoin SV        1,770,643,370 (  1.0%)
1